# Steel Door Manufacturer — Cost Analysis
## Supporting Analysis for Observations Tab
**Data:** 12 months of purchase orders (April 2023 – March 2024)  
**Client:** Steel door manufacturer, $300M revenue, single vendor, no contract  
**Material:** Galvalume steel coils (various thicknesses and paint finishes)

In [ ]:
# Run this cell first to ensure dependencies are installed in the active kernel
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'numpy', 'openpyxl', '-q'])

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the data (skip the header rows in the Excel file)
df = pd.read_excel('Manufacturing Case Study.xlsx', sheet_name='Data', skiprows=3, header=None)

# Assign proper column names
df.columns = ['_','Invoice_Num','Invoice_Date','Ship_Date','Ship_To_City','PO_Num','Part_Num',
              'Material_Description','Paint_Top','Paint_Bottom','Linear_Feet','Weight_lbs',
              'Material_Unit_Rate','Emboss_Unit_Rate','Freight_Unit_Rate','Fuel_Surcharge_Pct',
              'Total_Unit_Rate','Product_Expense','Emboss_Expense','Freight_Expense',
              'Subtotal_Expense','Fuel_Surcharge_Expense','Total_Expense']
df = df.drop(columns=['_'])

# Convert numeric columns
numeric_cols = ['Linear_Feet','Weight_lbs','Material_Unit_Rate','Emboss_Unit_Rate',
                'Freight_Unit_Rate','Fuel_Surcharge_Pct','Total_Unit_Rate','Product_Expense',
                'Emboss_Expense','Freight_Expense','Subtotal_Expense','Fuel_Surcharge_Expense',
                'Total_Expense']
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Convert dates
df['Invoice_Date'] = pd.to_datetime(df['Invoice_Date'], errors='coerce')
df['Ship_Date'] = pd.to_datetime(df['Ship_Date'], errors='coerce')

# Clean: drop rows without expense data, remove stray header rows
df = df.dropna(subset=['Total_Expense'])
df = df[df['Ship_To_City'] != 'Ship To City']

# Add month column for time-series analysis
df['Month'] = df['Invoice_Date'].dt.to_period('M')

print(f"Dataset: {df.shape[0]:,} line items across {df['Invoice_Num'].nunique()} invoices")
print(f"Date range: {df['Invoice_Date'].min().strftime('%B %Y')} to {df['Invoice_Date'].max().strftime('%B %Y')}")
print(f"Locations: {df['Ship_To_City'].nunique()} cities")
print(f"Part numbers: {df['Part_Num'].nunique()} unique parts")

---
## Section A: Spending Overview
*Charts: Monthly Expense by Category (stacked bar) + Expense Breakdown (pie)*

In [ ]:
# Total expense breakdown by category
total_spend = df['Total_Expense'].sum()
material = df['Product_Expense'].sum()
freight = df['Freight_Expense'].sum()
emboss = df['Emboss_Expense'].sum()
fuel = df['Fuel_Surcharge_Expense'].sum()

print("=" * 60)
print("TOTAL EXPENSE COMPOSITION")
print("=" * 60)
print(f"  Material:        ${material:>14,.0f}   ({material/total_spend*100:.1f}%)")
print(f"  Freight:         ${freight:>14,.0f}   ({freight/total_spend*100:.1f}%)")
print(f"  Emboss:          ${emboss:>14,.0f}   ({emboss/total_spend*100:.1f}%)")
print(f"  Fuel Surcharge:  ${fuel:>14,.0f}   ({fuel/total_spend*100:.1f}%)")
print(f"  {'─' * 45}")
print(f"  TOTAL:           ${total_spend:>14,.0f}   (100.0%)")
print(f"\n  Freight + Fuel combined: ${freight + fuel:,.0f} ({(freight+fuel)/total_spend*100:.1f}%)")

In [ ]:
# Monthly expense trend by category
monthly = df.groupby('Month').agg(
    Material=('Product_Expense', 'sum'),
    Emboss=('Emboss_Expense', 'sum'),
    Freight=('Freight_Expense', 'sum'),
    Fuel=('Fuel_Surcharge_Expense', 'sum'),
    Total=('Total_Expense', 'sum'),
    Weight=('Weight_lbs', 'sum'),
    Line_Items=('Invoice_Num', 'count'),
    Invoices=('Invoice_Num', 'nunique')
).reset_index()

monthly['Avg_Rate_Per100'] = monthly['Total'] / monthly['Weight'] * 100

print("MONTHLY EXPENSE TREND")
print("=" * 110)
print(f"{'Month':<10} {'Material':>12} {'Freight':>12} {'Emboss':>10} {'Fuel':>10} {'Total':>14} {'Weight (lbs)':>14} {'Rate/100':>10}")
print("─" * 110)
for _, r in monthly.iterrows():
    print(f"{str(r['Month']):<10} ${r['Material']:>11,.0f} ${r['Freight']:>11,.0f} ${r['Emboss']:>9,.0f} ${r['Fuel']:>9,.0f} ${r['Total']:>13,.0f} {r['Weight']:>13,.0f} ${r['Avg_Rate_Per100']:>8.2f}")

# Seasonality metrics
peak = monthly.loc[monthly['Weight'].idxmax()]
trough = monthly.loc[monthly['Weight'].idxmin()]
print(f"\nPeak month: {peak['Month']} — {peak['Weight']:,.0f} lbs (${peak['Total']:,.0f})")
print(f"Trough month: {trough['Month']} — {trough['Weight']:,.0f} lbs (${trough['Total']:,.0f})")
print(f"Peak-to-trough ratio: {peak['Weight']/trough['Weight']:.0f}:1")

---
## Section B: Location & Freight Analysis
*Charts: Freight & Blended Rate by Location (horizontal bar) + Total Spend by Location (bar)*

In [ ]:
# Expense breakdown by ship-to location
city = df.groupby('Ship_To_City').agg(
    Total=('Total_Expense', 'sum'),
    Weight=('Weight_lbs', 'sum'),
    Freight=('Freight_Expense', 'sum'),
    Fuel=('Fuel_Surcharge_Expense', 'sum'),
    Material=('Product_Expense', 'sum'),
    Emboss=('Emboss_Expense', 'sum'),
    Line_Items=('Invoice_Num', 'count'),
    Invoices=('Invoice_Num', 'nunique')
).reset_index()

city['Blended_Rate'] = (city['Total'] / city['Weight'] * 100).round(2)
city['Freight_Rate'] = (city['Freight'] / city['Weight'] * 100).round(2)
city['Freight_Pct'] = (city['Freight'] / city['Total'] * 100).round(1)
city['Order_Pct'] = (city['Line_Items'] / city['Line_Items'].sum() * 100).round(1)
city = city.sort_values('Total', ascending=False)

print("LOCATION ANALYSIS")
print("=" * 130)
print(f"{'City':<18} {'Total Spend':>14} {'Weight':>14} {'Blended Rate':>14} {'Freight Rate':>14} {'Freight $':>12} {'Frgt %':>8} {'Orders %':>9}")
print("─" * 130)
for _, r in city.iterrows():
    print(f"{r['Ship_To_City']:<18} ${r['Total']:>13,.0f} {r['Weight']:>13,.0f} ${r['Blended_Rate']:>12.2f} ${r['Freight_Rate']:>12.2f} ${r['Freight']:>11,.0f} {r['Freight_Pct']:>7.1f}% {r['Order_Pct']:>7.1f}%")

# Key findings
nash = city[city['Ship_To_City'] == 'Nashville, TN'].iloc[0]
sac = city[city['Ship_To_City'] == 'Sacramento, CA'].iloc[0]
print(f"\nKey gap: Nashville freight ${nash['Freight_Rate']:.2f}/100 lbs vs Sacramento ${sac['Freight_Rate']:.2f}/100 lbs")
print(f"Nashville freight cost: ${nash['Freight']:,.0f} ({nash['Freight']/df['Freight_Expense'].sum()*100:.0f}% of all freight)")
print(f"Blended rate gap: Nashville ${nash['Blended_Rate']:.2f} vs Sacramento ${sac['Blended_Rate']:.2f} = ${nash['Blended_Rate']-sac['Blended_Rate']:.2f} difference")

In [ ]:
# Freight rate statistics by city
print("FREIGHT RATE STATISTICS BY CITY")
print("=" * 80)
freight_stats = df.groupby('Ship_To_City')['Freight_Unit_Rate'].describe()
print(freight_stats.round(2).to_string())

# Potential savings from Nashville freight reduction
nash_weight = df[df['Ship_To_City'] == 'Nashville, TN']['Weight_lbs'].sum()
print(f"\nNashville total weight: {nash_weight:,.0f} lbs")
print(f"Savings if Nashville freight reduced by $2/100 lbs: ${nash_weight * 2 / 100:,.0f}")

---
## Section C: Part Number & Material Analysis
*Charts: Top 10 Parts table + Same-Thickness Rate Comparison (bar) + Order Size vs Rate (bar)*

In [ ]:
# Top parts by total expense
part = df.groupby('Part_Num').agg(
    Total=('Total_Expense', 'sum'),
    Weight=('Weight_lbs', 'sum'),
    Material=('Product_Expense', 'sum'),
    Emboss=('Emboss_Expense', 'sum'),
    Freight=('Freight_Expense', 'sum'),
    Fuel=('Fuel_Surcharge_Expense', 'sum'),
    Lines=('Invoice_Num', 'count'),
    Avg_Mat_Rate=('Material_Unit_Rate', 'mean'),
    Avg_Emboss_Rate=('Emboss_Unit_Rate', 'mean'),
    Avg_Freight_Rate=('Freight_Unit_Rate', 'mean')
).reset_index().sort_values('Total', ascending=False)

part['Pct'] = (part['Total'] / part['Total'].sum() * 100).round(1)
part['Cum_Pct'] = part['Pct'].cumsum().round(1)
part['Eff_Rate'] = (part['Total'] / part['Weight'] * 100).round(2)

print("TOP 10 PARTS BY TOTAL EXPENSE")
print("=" * 140)
print(f"{'Part #':<16} {'Weight':>12} {'Total Expense':>14} {'% Spend':>8} {'Cum %':>7} {'Mat Rate':>10} {'Freight Rate':>13} {'Eff Rate':>10} {'Lines':>6}")
print("─" * 140)
for _, r in part.head(10).iterrows():
    print(f"{r['Part_Num']:<16} {r['Weight']:>11,.0f} ${r['Total']:>13,.0f} {r['Pct']:>7.1f}% {r['Cum_Pct']:>6.1f}% ${r['Avg_Mat_Rate']:>8.2f} ${r['Avg_Freight_Rate']:>11.2f} ${r['Eff_Rate']:>8.2f} {int(r['Lines']):>5}")

top2_spend = part.head(2)['Total'].sum()
print(f"\nTop 2 parts combined: ${top2_spend:,.0f} = {top2_spend/part['Total'].sum()*100:.1f}% of total spend")

In [ ]:
# Identify Aluminum vs Galvalume parts
def classify_material(desc):
    if pd.isna(desc): return 'Unknown'
    return 'Aluminum' if 'Aluminum' in str(desc) else 'Galvalume'

df['Material_Type'] = df['Material_Description'].apply(classify_material)

mat = df.groupby('Material_Type').agg(
    Spend=('Total_Expense', 'sum'),
    Weight=('Weight_lbs', 'sum'),
    Avg_Mat_Rate=('Material_Unit_Rate', 'mean'),
    Lines=('Invoice_Num', 'count')
).reset_index()
mat['Eff_Rate'] = (mat['Spend'] / mat['Weight'] * 100).round(2)

print("ALUMINUM vs GALVALUME")
print("=" * 90)
print(f"{'Type':<14} {'Spend':>14} {'Weight':>14} {'Avg Mat Rate':>14} {'Eff Rate':>12} {'Lines':>7}")
print("─" * 90)
for _, r in mat.iterrows():
    print(f"{r['Material_Type']:<14} ${r['Spend']:>13,.0f} {r['Weight']:>13,.0f} ${r['Avg_Mat_Rate']:>12.2f} ${r['Eff_Rate']:>10.2f} {int(r['Lines']):>6}")

print("\n⚠ Aluminum parts (11-000067, 11-000055) cost 3.5× more per 100 lbs than Galvalume.")
print("  Evaluate whether Galvalume can substitute for these applications.")

In [ ]:
# Same-thickness (0.019") parts — rate varies by paint finish
thin = df[df['Material_Description'].str.contains('0.019', na=False)]

thin_parts = thin.groupby(['Part_Num', 'Paint_Top']).agg(
    Spend=('Total_Expense', 'sum'),
    Weight=('Weight_lbs', 'sum'),
    Rate=('Total_Unit_Rate', 'mean'),
    Lines=('Invoice_Num', 'count')
).reset_index().sort_values('Spend', ascending=False)

thin_parts['Eff_Rate'] = (thin_parts['Spend'] / thin_parts['Weight'] * 100).round(2)

print('SAME THICKNESS (0.019") PARTS — RATE BY PAINT FINISH')
print("=" * 120)
print(f"{'Part #':<14} {'Paint Finish':<35} {'Weight':>12} {'Spend':>14} {'Eff Rate':>10} {'Lines':>6}")
print("─" * 120)
for _, r in thin_parts.iterrows():
    paint = str(r['Paint_Top'])[:32]
    print(f"{r['Part_Num']:<14} {paint:<35} {r['Weight']:>11,.0f} ${r['Spend']:>13,.0f} ${r['Eff_Rate']:>8.2f} {int(r['Lines']):>5}")

# Savings calculation
lowest = thin_parts['Eff_Rate'].min()
total_weight_019 = thin_parts['Weight'].sum()
actual = thin_parts['Spend'].sum()
potential = total_weight_019 * lowest / 100
print(f"\nLowest rate: ${lowest:.2f}/100 lbs (Part 11-000406, Clear Resin)")
print(f"Highest rate: ${thin_parts['Eff_Rate'].max():.2f}/100 lbs")
print(f"Rate spread: ${thin_parts['Eff_Rate'].max() - lowest:.2f}/100 lbs")
print(f"\nIf all 0.019\" parts priced at lowest rate:")
print(f"  Actual spend:    ${actual:>14,.0f}")
print(f"  Potential spend: ${potential:>14,.0f}")
print(f"  Savings:         ${actual - potential:>14,.0f}")

In [ ]:
# Order size vs effective rate — do bigger orders get better pricing?
order = df.groupby('PO_Num').agg(
    Weight=('Weight_lbs', 'sum'),
    Spend=('Total_Expense', 'sum'),
    Lines=('Invoice_Num', 'count'),
    Parts=('Part_Num', 'nunique')
).reset_index()

order['Rate'] = order['Spend'] / order['Weight'] * 100
order['Size_Q'] = pd.qcut(order['Weight'], 4, labels=['Q1 - Small', 'Q2 - Medium', 'Q3 - Large', 'Q4 - Very Large'])

oq = order.groupby('Size_Q', observed=False).agg(
    Orders=('PO_Num', 'count'),
    Avg_Weight=('Weight', 'mean'),
    Min_Weight=('Weight', 'min'),
    Max_Weight=('Weight', 'max'),
    Avg_Rate=('Rate', 'mean'),
    Median_Rate=('Rate', 'median'),
    Total_Spend=('Spend', 'sum')
).reset_index()

print("ORDER SIZE vs EFFECTIVE RATE")
print("=" * 120)
print(f"{'Quartile':<18} {'Orders':>7} {'Avg Weight':>12} {'Weight Range':>22} {'Avg Rate':>10} {'Med Rate':>10} {'Total Spend':>14}")
print("─" * 120)
for _, r in oq.iterrows():
    rng = f"{r['Min_Weight']:,.0f}–{r['Max_Weight']:,.0f}"
    print(f"{r['Size_Q']:<18} {int(r['Orders']):>7} {r['Avg_Weight']:>11,.0f} {rng:>22} ${r['Avg_Rate']:>8.2f} ${r['Median_Rate']:>8.2f} ${r['Total_Spend']:>13,.0f}")

small_rate = oq.iloc[0]['Avg_Rate']
large_rate = oq.iloc[3]['Avg_Rate']
print(f"\nSmall order penalty: ${small_rate:.2f} - ${large_rate:.2f} = ${small_rate - large_rate:.2f}/100 lbs")
print("Consolidating into fewer, larger POs would reduce the effective rate immediately.")

---
## Section D: Rate Trends & Fuel Surcharges
*Charts: Material Rate Trend for Top 2 Parts (line) + Fuel Surcharge % by Month (line)*

In [ ]:
# Material rate trends for the two dominant parts
rate_447 = df[df['Part_Num'] == '11-000447'].groupby('Month')['Material_Unit_Rate'].agg(['mean', 'min', 'max']).reset_index()
rate_406 = df[df['Part_Num'] == '11-000406'].groupby('Month')['Material_Unit_Rate'].agg(['mean', 'min', 'max']).reset_index()

print("MATERIAL RATE TREND — PART 11-000447 (White Spectrascape)")
print("=" * 70)
print(f"{'Month':<10} {'Avg Rate':>10} {'Min':>10} {'Max':>10}")
print("─" * 70)
for _, r in rate_447.iterrows():
    print(f"{str(r['Month']):<10} ${r['mean']:>8.2f} ${r['min']:>8.2f} ${r['max']:>8.2f}")

print(f"\nPeak: ${rate_447['mean'].max():.2f} ({rate_447.loc[rate_447['mean'].idxmax(), 'Month']})")
print(f"Trough: ${rate_447['mean'].min():.2f} ({rate_447.loc[rate_447['mean'].idxmin(), 'Month']})")
print(f"Swing: {(rate_447['mean'].max() - rate_447['mean'].min()) / rate_447['mean'].min() * 100:.0f}%")

print(f"\n\nMATERIAL RATE TREND — PART 11-000406 (Clear Resin)")
print("=" * 70)
print(f"{'Month':<10} {'Avg Rate':>10} {'Min':>10} {'Max':>10}")
print("─" * 70)
for _, r in rate_406.iterrows():
    print(f"{str(r['Month']):<10} ${r['mean']:>8.2f} ${r['min']:>8.2f} ${r['max']:>8.2f}")

print(f"\nPeak: ${rate_406['mean'].max():.2f} ({rate_406.loc[rate_406['mean'].idxmax(), 'Month']})")
print(f"Trough: ${rate_406['mean'].min():.2f} ({rate_406.loc[rate_406['mean'].idxmin(), 'Month']})")
print(f"Swing: {(rate_406['mean'].max() - rate_406['mean'].min()) / rate_406['mean'].min() * 100:.0f}%")

In [ ]:
# Fuel surcharge analysis
fuel_monthly = df.groupby('Month')['Fuel_Surcharge_Pct'].agg(['mean', 'min', 'max', 'std']).reset_index()

print("FUEL SURCHARGE % BY MONTH")
print("=" * 70)
print(f"{'Month':<10} {'Avg %':>8} {'Min %':>8} {'Max %':>8} {'Std Dev':>8}")
print("─" * 70)
for _, r in fuel_monthly.iterrows():
    print(f"{str(r['Month']):<10} {r['mean']*100:>7.1f}% {r['min']*100:>7.1f}% {r['max']*100:>7.1f}% {r['std']*100:>7.2f}%")

print(f"\nOverall statistics:")
print(f"  Mean:   {df['Fuel_Surcharge_Pct'].mean()*100:.1f}%")
print(f"  Median: {df['Fuel_Surcharge_Pct'].median()*100:.1f}%")
print(f"  Max:    {df['Fuel_Surcharge_Pct'].max()*100:.1f}%  ← OUTLIER — investigate")
print(f"  Total fuel surcharge cost: ${df['Fuel_Surcharge_Expense'].sum():,.0f}")

# Savings from cap
current_fuel = df['Fuel_Surcharge_Expense'].sum()
# Estimate: if capped at 25%, what would have been saved on months above 25%
df['Capped_Fuel_Pct'] = df['Fuel_Surcharge_Pct'].clip(upper=0.25)
df['Capped_Fuel_Expense'] = df['Subtotal_Expense'] * df['Capped_Fuel_Pct']
capped_fuel = df['Capped_Fuel_Expense'].sum()
print(f"\nIf fuel surcharge capped at 25%:")
print(f"  Current fuel cost:  ${current_fuel:>12,.0f}")
print(f"  Capped fuel cost:   ${capped_fuel:>12,.0f}")
print(f"  Estimated savings:  ${current_fuel - capped_fuel:>12,.0f}")

---
## Summary of Key Numbers Used in Observations Tab

| Metric | Value | Source Cell Above |
|--------|-------|-------------------|
| Total Spend | $35.3M | Section A - Expense Composition |
| Material % | 87.8% | Section A - Expense Composition |
| Freight + Fuel | $3.2M (9.1%) | Section A - Expense Composition |
| Peak-to-trough volume ratio | 15:1 | Section A - Monthly Trend |
| Nashville freight rate | $9.48/100 lbs | Section B - Location Analysis |
| Sacramento freight rate | $0.46/100 lbs | Section B - Location Analysis |
| Nashville blended rate gap | $12.25/100 lbs vs Sacramento | Section B - Location Analysis |
| Nashville freight savings ($2 reduction) | ~$170K | Section B - Freight Stats |
| Top 2 parts % of spend | 91.3% | Section C - Top 10 Parts |
| Aluminum rate premium | 3.5× Galvalume | Section C - Aluminum vs Galvalume |
| 0.019" rate spread by paint | $20/100 lbs | Section C - Same Thickness |
| Potential savings (equalized 0.019" rate) | ~$1.5M | Section C - Same Thickness |
| Small order rate penalty | $12/100 lbs | Section C - Order Size |
| Material rate seasonal swing | 25–30% | Section D - Rate Trends |
| Fuel surcharge outlier | 329.5% | Section D - Fuel Surcharge |
| Fuel cap savings estimate | $100K–$200K | Section D - Fuel Surcharge |